In [1]:
import numpy as np
import os
import time
import re

from torch.utils.data import Dataset, DataLoader, Subset

import torch

import random

In [2]:
use_cuda = torch.cuda.is_available()

torch.manual_seed(143421)
if use_cuda:
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print(use_cuda)

False


In [3]:
path_to_file = "data/khayyam.txt"
# Read, then decode for py2 compat.
text = open(path_to_file, 'rb').read().decode(encoding='utf-8')
# length of text is the number of characters in it
print ('Length of text: {} characters'.format(len(text)))

Length of text: 454548 characters


In [4]:
print(text[-240:])

|در گو و در غارها مسکن گرفت
|ابر می بارید چون مشک اشکها
|حاجیان جمله گشاده مشکها
|یک جماعت زان عجایب کارها
|می بریدند از میان زنارها
|قوم دیگر را یقین در ازدیاد
|زین عجب والله اعلم بالرشاد
|قوم دیگر ناپذیرا ترش و خام
|ناقصان سرمدی تم الکلام


In [5]:
# The unique characters in the file
vocab = sorted(set(text))
print ('{} unique characters'.format(len(vocab)))

47 unique characters


In [6]:
def clean_persian(text):
    text = text.replace('ي', 'ی').replace('ك', 'ک')
    text = re.sub(r'[ًٌٍَُِّْ]', '', text)
    text = text.replace('\u200c', ' ')
    return text

In [7]:
clean_persian(text)

'|برخیز بتا بیا ز بهر دل ما\n|حل کن به جمال خویشتن مشکل ما\n|یک کوزه شراب تا به هم نوش کنیم\n|زآن پیش که کوزه ها کنند از گل ما\n|چون عهده نمی شود کسی فردا را\n|حالی خوش دار این دل پر سودا را\n|می نوش به ماهتاب ای ماه که ماه\n|بسیار بتابد و نیابد ما را\n|قرآن که مهین کلام خوانند آن را\n|گه گاه نه بر دوام خوانند آن را\n|بر گرد پیاله آیتی هست مقیم\n|کاندر همه جا مدام خوانند آن را\n|گر می نخوری طعنه مزن مستانرا\n|بنیاد مکن تو حیله و دستانرا\n|تو غره بدان مشو که می مینخوری\n|صد لقمه خوری که می غلام ست آنرا\n|هر چند که رنگ و بوی زیباست مرا\n|چون لاله رخ و چو سرو بالاست مرا\n|معلوم نشد که در طربخانه خاک\n|نقاش ازل بهر چه آراست مرا\n|ماییم و می و مطرب و این کنج خراب\n|جان و دل و جام و جامه پر درد شراب\n|فارغ ز امید رحمت و بیم عذاب\n|آزاد ز خاک و باد و از آتش و آب\n|آن قصر که جمشید در او جام گرفت\n|آهو بچه کرد و روبه آرام گرفت\n|بهرام که گور می گرفتی همه عمر\n|دیدی که چگونه گور بهرام گرفت\n|ابر آمد و باز بر سر سبزه گریست\n|بی بادهٔ گلرنگ نمی باید زیست\n|این سبزه که امروز تماشاگه ماست\n|تا سبزهٔ

In [8]:
class CharTokenizer:
    def __init__(self, texts):
        chars = sorted(set(''.join(texts)))
        self.stoi = {ch: i+1 for i, ch in enumerate(chars)}
        self.itos = {i+1: ch for i, ch in enumerate(chars)}
        self.vocab_size = len(chars) + 1

    def encode(self, text):
        return [self.stoi[ch] for ch in text if ch in self.stoi]

    def decode(self, indices):
        return ''.join([self.itos.get(i, '') for i in indices if i != 0])

In [9]:
tokenizer = CharTokenizer(texts=text)

print(tokenizer.encode("سلام"))
print(tokenizer.decode([22, 32, 10, 33]))


[22, 32, 10, 33]
سلام


In [10]:
text_int = tokenizer.encode(text)
text_int[:5]

[4, 11, 20, 17, 47]

In [11]:
print ('{} ---- characters mapped to int ---- > {}'.format(repr(text[:58]), text_int[:58]))

'|برخیز بتا بیا ز بهر دل ما\n|حل کن به جمال خویشتن مشکل ما\n|' ---- characters mapped to int ---- > [4, 11, 20, 17, 47, 21, 2, 11, 13, 10, 2, 11, 47, 10, 2, 21, 2, 11, 35, 20, 2, 18, 32, 2, 33, 10, 1, 4, 16, 32, 2, 45, 34, 2, 11, 35, 2, 15, 33, 10, 32, 2, 17, 36, 47, 23, 13, 34, 2, 33, 23, 45, 32, 2, 33, 10, 1, 4]


In [12]:
class TextDataset(Dataset):
    def __init__(self, text, seq_len, tokenizer):
        self.seq_len = seq_len
        self.data = tokenizer.encode(text)
        self.vocab_size = tokenizer.vocab_size

    def __len__(self):
        return len(self.data) - self.seq_len

    def __getitem__(self, idx):
        x = self.data[idx: idx + self.seq_len]
        y = self.data[idx + 1: idx + self.seq_len + 1]
        return torch.tensor(x), torch.tensor(y)
    
train_loader = DataLoader(text, batch_size=32, shuffle=True)


In [13]:
import torch.nn as nn

class GRUModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=32, hidden_dim=64, num_layers=2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        
        self.gru = nn.GRU(embed_dim, hidden_dim, num_layers, batch_first=True)
        
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        emb = self.embed(x)
        
        out, _ = self.gru(emb) 
        
        return self.fc(out)  # (batch, seq_len, vocab_size)


In [14]:
import torch
import torch.nn as nn

class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size, d_model=256, nhead=8, num_layers=4,
                 dim_feedforward=1024, dropout=0.2, max_len=512):
        super().__init__()
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=dim_feedforward, dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers)

        self.ln = nn.LayerNorm(d_model)
        self.fc = nn.Linear(d_model, vocab_size)
        self.max_len = max_len
        self.d_model = d_model

    def forward(self, x):
        # x: (B, T)
        B, T = x.shape
        pos = torch.arange(T, device=x.device).unsqueeze(0)  # (1, T)

        tok_emb = self.token_emb(x) * (self.d_model ** 0.5)
        pos_emb = self.pos_emb(pos)
        h = tok_emb + pos_emb  # (B, T, d_model)

        # causal mask
        mask = torch.triu(torch.ones(T, T, device=x.device), diagonal=1).bool()

        out = self.transformer(h, mask=mask)  # (B, T, d_model)
        out = self.ln(out)
        logits = self.fc(out)  # (B, T, vocab)
        return logits

In [15]:
def train(model, loader, opt, crit, device):
    model.train()
    total = 0.0

    for step, (x, y) in enumerate(loader, start=1):
        x, y = x.to(device), y.to(device)

        opt.zero_grad()
        logits = model(x)  # (B, T, vocab)
        loss = crit(logits.view(-1, model.fc.out_features), y.view(-1))
        loss.backward()
        opt.step()

        total += loss.item()
        if step == 1 or step % 20 == 0:
            print(f"step {step}/{len(loader)} | loss: {loss.item():.6f} | avg: {total/step:.6f}")

    return total / len(loader)

In [16]:
import torch
import matplotlib.pyplot as plt

def train_with_early_stopping(
    model, train_loader, val_loader, opt, crit, device,
    epochs=50, patience=5, min_delta=1e-4, grad_clip=1.0, print_every=10
):
    best_val = float("inf")
    bad_epochs = 0
    best_state = None

    train_losses = []
    val_losses = []

    for epoch in range(epochs):
        # ===================== TRAIN =====================
        model.train()
        train_loss = 0.0

        print(f"\n=== Epoch {epoch+1}/{epochs} ===")

        for batch_idx, (x, y) in enumerate(train_loader, start=1):
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)

            opt.zero_grad(set_to_none=True)
            logits = model(x)
            loss = crit(logits.view(-1, model.fc.out_features), y.view(-1))
            loss.backward()

            if grad_clip is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

            opt.step()
            train_loss += loss.item()

            # نمایش روند داخل هر epoch
            if batch_idx % print_every == 0 or batch_idx == len(train_loader):
                avg_so_far = train_loss / batch_idx
                print(f"  [Train] Batch {batch_idx}/{len(train_loader)} | "
                      f"loss: {loss.item():.4f} | avg_loss_so_far: {avg_so_far:.4f}")

        train_loss /= len(train_loader)

        # ===================== VALIDATION =====================
        model.eval()
        val_loss = 0.0

        with torch.no_grad():
            for batch_idx, (x, y) in enumerate(val_loader, start=1):
                x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
                logits = model(x)
                loss = crit(logits.view(-1, model.fc.out_features), y.view(-1))
                val_loss += loss.item()

                if batch_idx % print_every == 0 or batch_idx == len(val_loader):
                    avg_so_far = val_loss / batch_idx
                    print(f"  [Val]   Batch {batch_idx}/{len(val_loader)} | "
                          f"loss: {loss.item():.4f} | avg_loss_so_far: {avg_so_far:.4f}")

        val_loss /= len(val_loader)

        train_losses.append(train_loss)
        val_losses.append(val_loss)

        print(f"Epoch {epoch+1} done | train_loss: {train_loss:.4f} | val_loss: {val_loss:.4f}")

        # ===================== EARLY STOPPING =====================
        if val_loss < best_val - min_delta:
            best_val = val_loss
            bad_epochs = 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            print(f"  ✅ New best val loss: {best_val:.4f}")
        else:
            bad_epochs += 1
            print(f"  ⚠ No improvement ({bad_epochs}/{patience})")

            if bad_epochs >= patience:
                print(f"Early stopping! Best val loss: {best_val:.4f}")
                if best_state is not None:
                    model.load_state_dict(best_state)
                break

    # ===================== PLOT =====================
    plt.figure(figsize=(9, 5))
    plt.plot(range(1, len(train_losses) + 1), train_losses, marker="o", label="Train Loss")
    plt.plot(range(1, len(val_losses) + 1), val_losses, marker="o", label="Val Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training / Validation Loss")
    plt.grid(True)
    plt.legend()
    plt.tight_layout()
    plt.show()

    return model, train_losses, val_losses


In [17]:
def greedy_sample(logits):
    return torch.argmax(logits, dim=-1)

In [18]:
def sample_with_temp(logits, temperature=0.8):
    probs = torch.softmax(logits / temperature, dim=-1)
    return torch.multinomial(probs, num_samples=1)

In [19]:
def sample_with_temp(logits, temperature=0.8):
    probs = torch.softmax(logits / temperature, dim=-1)
    return torch.multinomial(probs, num_samples=1)

In [20]:
def top_k_sample(logits, k=40):
    values, indices = torch.topk(logits, k, dim=-1)
    min_val = values[:, -1].unsqueeze(-1)
    logits[logits < min_val] = float('-inf')
    probs = torch.softmax(logits, dim=-1)
    return torch.multinomial(probs, num_samples=1)

In [21]:
def top_p_sample(logits, p=0.9):
    sorted_logits, sorted_idx = torch.sort(logits, descending=True)
    sorted_probs = torch.softmax(sorted_logits, dim=-1)
    cum_probs = torch.cumsum(sorted_probs, dim=-1)
    mask = cum_probs - sorted_probs > p
    sorted_logits[mask] = float('-inf')
    logits = torch.zeros_like(logits).scatter_(-1, sorted_idx, sorted_logits)
    probs = torch.softmax(logits, dim=-1)
    return torch.multinomial(probs, num_samples=1)

In [22]:
@torch.no_grad()
def generate(model, tokenizer, prompt, max_new_tokens=50, strategy='top_p',
             temperature=0.8, top_k=40, top_p=0.9, device='cpu'):
    model.eval()
    idx = torch.tensor([tokenizer.encode(prompt)], device=device)
    for _ in range(max_new_tokens):
        if hasattr(model, 'max_len'):
            idx_crop = idx[:, -model.max_len:]
        else:
            idx_crop = idx

        logits = model(idx_crop)
        if isinstance(logits, tuple):
            logits = logits[0]
        logits = logits[:, -1, :]

        if strategy == 'greedy':
            next_token = greedy_sample(logits)
        elif strategy == 'temperature':
            next_token = sample_with_temp(logits, temperature)
        elif strategy == 'top_k':
            next_token = top_k_sample(logits, top_k)
        elif strategy == 'top_p':
            next_token = top_p_sample(logits, top_p)
        else:
            raise ValueError("Unknown strategy")

        idx = torch.cat([idx, next_token], dim=-1)

    generated = tokenizer.decode(idx[0].tolist())
    return generated[len(prompt):]

In [24]:
model = GRUModel(vocab_size=tokenizer.vocab_size,
                          embed_dim=64, hidden_dim=128).to(device)
criterion = nn.CrossEntropyLoss(ignore_index=0)
optimizer = torch.optim.Adam(model.parameters(), lr=0.1)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.4)

In [25]:
def split_dataset(dataset, val_ratio=0.1, seed=143421):
    n = len(dataset)
    idx = list(range(n))
    random.Random(seed).shuffle(idx)
    n_val = int(n * val_ratio)
    val_idx = idx[:n_val]
    train_idx = idx[n_val:]
    return Subset(dataset, train_idx), Subset(dataset, val_idx)

In [ ]:
all_text = open(path_to_file, encoding='utf-8').read()
all_text = clean_persian(all_text)
tokenizer = CharTokenizer([all_text])


dataset = TextDataset(all_text, seq_len=40, tokenizer=tokenizer)
loader = DataLoader(dataset, batch_size=64, shuffle=True)

train_ds, val_ds = split_dataset(dataset, val_ratio=0.1)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, num_workers=2, pin_memory=use_cuda)
val_loader   = DataLoader(val_ds, batch_size=64, shuffle=False, num_workers=2, pin_memory=use_cuda)

train_with_early_stopping(
    model, train_loader, val_loader,
    optimizer, criterion, device,
    epochs=10, patience=4, min_delta=1e-4, grad_clip=1.0
)

print(generate(model, tokenizer, "الا یا ایها الساقی ادر کأساً و ناولها", max_new_tokens=50))


=== Epoch 1/10 ===
